# Google One AI Kit — 00. 初始化云端目录与下载底模（免费 CPU 运行）

> **重要省钱提示**：本 Notebook 只做 Google Drive 建目录和下载模型文件，**请务必使用免费 CPU 运行时**！  
> - 顶部菜单确认：**Runtime -> Change runtime type -> CPU**（消耗 **0** 计算单元）。  
> - 点击右上角 `Connect` 旁下拉箭头 -> **View resources** 可查看你的 Google AI Pro 每月 200 计算单元余额。

In [ ]:
# @title 1. 检查运行时类型与磁盘空间（确保处于 CPU 模式）
import shutil

try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"⚠️ 警告：当前检测到 GPU ({gpu_name})，下载期间会消耗计算单元！")
        print("👉 建议立即点击：Runtime -> Change runtime type -> CPU，然后重新连接。")
    else:
        print("✅ 当前为免费 CPU 运行时，不消耗 Colab 计算单元。")
except ImportError:
    print("✅ 当前为免费 CPU 运行时，不消耗 Colab 计算单元。")

total, used, free = shutil.disk_usage("/content")
print(f"/content 临时磁盘可用空间: {free / (1024**3):.1f} GB")

In [ ]:
# @title 2. 挂载 Google Drive 并初始化 GoogleOneAI_Studio 目录结构
import os
import shutil
from google.colab import drive

drive.mount("/content/drive")

STUDIO_DIR = "/content/drive/MyDrive/GoogleOneAI_Studio"
SUBDIRS = [
    "notebooks",
    "models/checkpoints",
    "models/style_models",
    "models/clip_vision",
    "models/loras",
    "models/diffusion_models",
    "models/text_encoders",
    "models/vae",
    "workflows",
    "input_refs",
    "datasets/custom_style_v1",
    "outputs",
    "training_runs",
]

for sub in SUBDIRS:
    os.makedirs(os.path.join(STUDIO_DIR, sub), exist_ok=True)

total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"✅ 目录初始化完成: {STUDIO_DIR}")
print(f"Google Drive 剩余空间: {free / (1024**3):.1f} GB / 总计 {total / (1024**3):.1f} GB")
if free < 25 * (1024**3):
    print("⚠️ Drive 剩余空间不足 25 GB，请先清理后再下载 FLUX 模型。")

In [ ]:
# @title 3. 安装 Hugging Face 高速下载组件 (huggingface_hub + hf_transfer)
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub[hf_transfer]"], check=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("✅ 下载组件准备就绪")

In [ ]:
# @title 4. 拉取 FLUX.1-dev FP8 + Redux + SigLIP 到 Google Drive（约 19 GB，无需 Token）
import os
from huggingface_hub import hf_hub_download

STUDIO_DIR = "/content/drive/MyDrive/GoogleOneAI_Studio"
MODELS = [
    ("Comfy-Org/Flux-1-Dev", "flux1-dev-fp8.safetensors", f"{STUDIO_DIR}/models/checkpoints"),
    ("Comfy-Org/Flux-1-Redux-Dev", "flux1-redux-dev.safetensors", f"{STUDIO_DIR}/models/style_models"),
    ("Comfy-Org/sigclip_vision_384", "sigclip_vision_patch14_384.safetensors", f"{STUDIO_DIR}/models/clip_vision"),
]

MIN_VALID_BYTES = 50 * 1024 * 1024  # 50 MB

for repo_id, filename, target_dir in MODELS:
    os.makedirs(target_dir, exist_ok=True)
    target_path = os.path.join(target_dir, filename)
    if os.path.exists(target_path) and os.path.getsize(target_path) > MIN_VALID_BYTES:
        size_gb = os.path.getsize(target_path) / (1024**3)
        print(f"⏭️ 已存在，跳过下载: {filename} ({size_gb:.2f} GB)")
        continue

    print(f"⬇️ 正在下载 {repo_id}/{filename} -> {target_dir} ...")
    downloaded = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=target_dir,
    )
    size_gb = os.path.getsize(downloaded) / (1024**3)
    print(f"✅ 下载完成: {downloaded} ({size_gb:.2f} GB)")

print("\n=== 模型校验结果 ===")
all_ok = True
for _, filename, target_dir in MODELS:
    p = os.path.join(target_dir, filename)
    if os.path.exists(p) and os.path.getsize(p) > MIN_VALID_BYTES:
        print(f"  [OK] {p} ({os.path.getsize(p) / (1024**3):.2f} GB)")
    else:
        print(f"  [MISSING] {p}")
        all_ok = False

if not all_ok:
    raise RuntimeError("部分模型未完整下载，请重新运行当前单元格。")
print("\n✅ 模型文件校验通过，继续运行第 5 步完成云盘强制刷盘！")

In [ ]:
# @title 5. 强制刷盘并安全卸载 Drive（防止 16GB 大模型未同步即关机丢失）
from google.colab import drive

print("⏳ 正在将 19 GB 模型缓存彻底刷盘写入 Google Drive（可能需要 30~90 秒，请勿关闭页面）...")
drive.flush_and_unmount()
print("🎉 全部底模已安全永久落盘到 Google Drive！")
print("👉 下一步：可以放心关闭本页（Runtime -> Disconnect and delete runtime），打开 01_comfyui_studio.ipynb（L4 GPU）开始出图。")